In [5]:
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.document_converter import DocumentConverter
import tqdm
import os
import glob
import re
import pandas as pd
from sentence_splitter import split_text_into_sentences

In [10]:
texts_path = "../../data/TEXT_stoxx600_docling"
filterd_texts_path = "../data/FILTERED_TEXT_stoxx600_docling"

In [13]:
texts = glob.glob(texts_path + "/*.txt")

In [24]:
for text_path in tqdm.tqdm(texts): 
    accepted_lines = lines
    not_accepted_lines = []

    with open(text_path, "r") as f: 
        text = f.read()
    lines = text.split("\n")

    is_image = lambda line: line == '<!-- image -->'
    is_table = lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False
    
    accepted_lines = [line for line in accepted_lines if not is_image(line) and not is_table(line)]
    not_accepted_lines = [line for line in accepted_lines if is_image(line) or is_table(line)]

    df = pd.DataFrame(accepted_lines, columns=["lines"])
    df = df.drop_duplicates()
    df["len"] = df["lines"].apply(len)
    df = df.sort_values("len")
    
    df.to_csv(os.path.join(filterd_texts_path, os.path.basename(text_path)[:-4]) + "_lines_len.csv")

100%|██████████| 42/42 [00:00<00:00, 102.09it/s]


In [25]:
filter_len = lambda line: len(line) > 110
accepted_lines = [line for line in accepted_lines if filter_len(line)]

In [ ]:
chunks = []

for line in accepted_lines: 
    sentences = split_text_into_sentences(line, language='en')
    sentences = [sentence.strip() for sentence in sentences]
    sentences = [sentence for sentence in sentences if sentence != ""]
    new_chunks = [(" ".join(sentences[i:i+3])).strip() for i in range(0, len(sentences), 3)]

    chunks += new_chunks 

[print(chunk) for chunk in chunks]

Euronext N.V. (the 'Company' or 'Euronext' and together with its subsidiaries, the 'Group') is a Dutch public company with limited liability (naamloze vennootschap), whose ordinary shares are admitted to listing and trading on regulated markets  in  the  Netherlands, France,  Belgium  and  Portugal.  The  applicable  regulations  with  respect  to public information and protection of investors, as well as the commitments made by the Company to securities and market authorities, are described in this Universal Registration Document (the 'Universal Registration Document').
["Euronext N.V. (the 'Company' or 'Euronext' and together with its subsidiaries, the 'Group') is a Dutch public company with limited liability (naamloze vennootschap), whose ordinary shares are admitted to listing and trading on regulated markets in the Netherlands, France, Belgium and Portugal.", "The applicable regulations with respect to public information and protection of investors, as well as the commitments made

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

# Try new filter, that only contains full sentence. 

In [70]:
lines = []

for i in range(10): 
    with open(texts[i], "r") as f: 
            text = f.read()
    lines += text.split("\n")

In [71]:
lines = list(set(lines))

In [117]:
lines = ["## The 15 elements of Yara's Compliance Program",
"1. Culture and tone at the top: Strong, explicit, and visible support and commitment to our policies from Yara's directors and senior management.",
"2. Risk management/periodic risk-based review: Clearly defined responsibilities and authorities for the implementation and oversight of the Compliance Program and codes, policies, and procedures, and for reporting to independent monitoring bodies.",
"3. Compliance organization / proper oversight, independence and resources: Clearly defined responsibilities and authorities for the implementation and oversight of the Compliance Program and codes, policies, and procedures, and for reporting to independent monitoring bodies.",
]

In [118]:
accepted_lines = []
not_accepted_lines = []

is_image = lambda line: line == '<!-- image -->'
is_table = lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False

# drop if condidtion is True
conditions = [
    lambda line: line == '<!-- image -->',
    lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 
    lambda line: line.strip()[0] == "#" if len(line) > 0 else True,
    lambda line: "." not in line,
    
    # more than 50% is numbers
    lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

    # minimum 3 words 
    lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

    # Minimum 2 Sentences
    #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

]
accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
not_accepted_lines = [line for line in lines if any(condition(line) for condition in conditions)]

In [119]:
# Minimum 2 Sentences
min_2_sentence = lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

min_2_sentence("München den 16. März 2023")

True

In [120]:
df_accepted_lines = pd.DataFrame(accepted_lines, columns=["Line"])
df_accepted_lines["len"] = df_accepted_lines["Line"].apply(len)
df_accepted_lines.sort_values("len")

,Line,len
0,"1. Culture and tone at the top: Strong, explic...",145
1,2. Risk management/periodic risk-based review:...,247
2,"3. Compliance organization / proper oversight,...",275


In [121]:
df_not_accepted_lines = pd.DataFrame(not_accepted_lines, columns=["Line"])
df_not_accepted_lines["len"] = df_not_accepted_lines["Line"].apply(len)
df_not_accepted_lines.sort_values("len")

,Line,len
0,## The 15 elements of Yara's Compliance Program,47


In [125]:
sentence_length = 3

for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]
new_chunks

['3. Compliance organization / proper oversight, independence and resources: Clearly defined responsibilities and authorities for the implementation and oversight of the Compliance Program and codes, policies, and procedures, and for reporting to independent monitoring bodies.']